# nb6 — Bilayer model variants on the Sec. 5 quartz/Si data (loop-1 checks C3 and C4)

Does the negative $\kappa_{\rm Si}$ of the frequency-independent quartz/Si fit (nb3) disappear when the model is changed?

* **C3 — dispersion.** The six-knot model of nb5 (`dispersive_grid64`, knots 0.2…3 THz, natural cubic spline), two layers
  (quartz / Si, no gap), **no hinge penalty** ($\lambda_\kappa = 0$):
  (i) the nb5 protocol with Adam (MAE + log-cosh; stage 1 knots tied, stage 2 knots free);
  (ii) the converged least-squares comparison: rMSE L-BFGS of the tied (frequency-independent) and of the six-knot model,
  the latter warm-started from the former, so that both end points are optimizer-independent.
* **C4 — interface layer.** Three-layer frequency-independent model quartz / gap / Si with rMSE L-BFGS from the three starts of nb3:
  (a) air gap ($n = 1$, $\kappa = 0$, $d_{\rm gap}$ free, starts 1, 5, 20 µm); (b) adhesive layer ($n_g \in [1.4, 1.8]$, $\kappa_g = 0$,
  $d_g$ free, same starts); (c) the two-layer model with a free reference delay τ and amplitude $a$; baseline = two-layer rMSE.

Outputs: `results/nb6.json` (no figure).

In [1]:
import sys, os, json, time
sys.path.insert(0, '../code')
import numpy as np
import torch
torch.set_num_threads(int(os.environ.get('NB_THREADS', 2)))
torch.set_default_dtype(torch.float64)
from quartzsi_model import QuartzSiProblem, STARTS, PARAM_NAMES, DT
from dispersive_grid64 import Grid, DispersiveMultiLayerGrid, interp_basis, fit, simulate_dispersive
from loop1_tools import lbfgs_minimize, grad_norm, tau_amp_filter

import paperstyle as ps
os.makedirs('../results', exist_ok=True)
P = QuartzSiProblem('../data/Reference_15min_nitrogen.txt', '../data/Quartz_Si_15min_nitrogen.txt', prepulse_ps=10.0)
L, deltat, sigma = P.L, P.deltat, P.sigma
x_exp, y_meas = P.x_exp, P.true_x_exp
RESULTS = dict(notebook='nb6_bilayer_model_variants.ipynb', data=dict(reference='data/Reference_15min_nitrogen.txt', sample='data/Quartz_Si_15min_nitrogen.txt',
                                                                          L=L, dt_fs=deltat * 1e15, sigma_prepulse=sigma))
T = {}
print(f'L = {L}, dt = {deltat*1e15:.1f} fs, sigma = {sigma:.3e}')

L = 2425, dt = 33.0 fs, sigma = 2.901e-04


## C3 (i) — six knots, nb5 protocol with Adam, λ_κ = 0

Stage 1 (knots tied) and stage 2 (knots free).  At lr = 1e-4 Adam moves κ by at most 5e-7 per iteration, so a 10 000-iteration
stage 1 cannot bring κ_Si from its start 0.0054 to the nb3 value; each stage therefore runs a short lr = 1e-3 phase followed by
lr = 1e-4 (stage 1: 6000 + 4000 iterations; stage 2: 6000 + 14 000).

In [2]:
F_LO, F_HI, N_KNOTS = 0.2, 3.0, 6
grid = Grid(L, deltat, f_lo=F_LO, f_hi=F_HI)
f_knots = np.linspace(F_LO, F_HI, N_KNOTS)
specs = [dict(name='quartz', n=1.9820, k=0.0071, d=1.0e-3), dict(name='Si', n=3.4000, k=0.0054, d=0.4e-3)]
SCHED1, SCHED2 = [(6000, 1e-3), (4000, 1e-4)], [(6000, 1e-3), (14000, 1e-4)]
def run_stage(m, sched, label):
    hist = []
    for n_it, lr in sched:
        h = fit(m, x_exp, y_meas, n_iters=n_it, lr=lr, print_step=5000, label=f'{label} lr {lr:g}', k_pos_weight=0.0)
        hist.append(h); m.load_state(m.state())
    return hist
t0 = time.time()
mA = DispersiveMultiLayerGrid(grid, specs, n_knots=N_KNOTS, interp='cubic', n_scale=0.1, k_scale=0.005); mA.set_thickness(True); mA.set_dispersion(False)
h1 = run_stage(mA, SCHED1, 'C3 stage 1 (tied)'); st1 = mA.state()
mA.set_dispersion(True)
h2 = run_stage(mA, SCHED2, 'C3 stage 2 (6 knots)'); st2 = mA.state()
T['C3_adam_s'] = time.time() - t0
def y_model(m):
    with torch.no_grad():
        return m.forward(x_exp)[1].numpy()
c3a = dict(protocol='Adam, MAE + log-cosh, lambda_kappa = 0; stage 1 ' + ' + '.join(f'{n} it. at lr {lr:g}' for n, lr in SCHED1) +
                    '; stage 2 ' + ' + '.join(f'{n} it. at lr {lr:g}' for n, lr in SCHED2), wall_time_s=T['C3_adam_s'],
           stage1=dict(n_quartz=st1['n'][0, 0].item(), k_quartz=st1['k'][0, 0].item(), d_quartz_um=st1['d'][0].item() * 1e6,
                       n_Si=st1['n'][1, 0].item(), k_Si=st1['k'][1, 0].item(), d_Si_um=st1['d'][1].item() * 1e6, loss=float(h1[-1]['loss'][-1])),
           stage2=dict(f_knots_THz=f_knots, quartz_n=st2['n'][0].numpy(), quartz_k=st2['k'][0].numpy(), Si_n=st2['n'][1].numpy(), Si_k=st2['k'][1].numpy(),
                       d_quartz_um=st2['d'][0].item() * 1e6, d_Si_um=st2['d'][1].item() * 1e6, loss=float(h2[-1]['loss'][-1]),
                       Si_k_negative_fraction_in_band=float(mA.k_negative_fraction()[1]),
                       Si_k_mean_interior_knots=float(st2['k'][1].numpy()[1:-1].mean()),
                       loss_last_2000_range=[float(h2[-1]['loss'][-2000:].min()), float(h2[-1]['loss'][-2000:].max())],
                       Si_k_knots_2000_iterations_before_end=h2[-1]['k'][-2000][1]))
print(f"\nC3 (i), {T['C3_adam_s']:.0f} s.  stage 1: kappa_Si {c3a['stage1']['k_Si']:+.3e}, kappa_q {c3a['stage1']['k_quartz']:.3e}, d_q {c3a['stage1']['d_quartz_um']:.2f}, "
      f"d_Si {c3a['stage1']['d_Si_um']:.2f} um, loss {c3a['stage1']['loss']:.5e}")
print('  stage 2: Si kappa knots ' + ', '.join(f'{v:+.5f}' for v in c3a['stage2']['Si_k']) + f"  (negative in band {100*c3a['stage2']['Si_k_negative_fraction_in_band']:.0f} %)")
print('           Si n knots     ' + ', '.join(f'{v:.4f}' for v in c3a['stage2']['Si_n']))
print('           quartz k knots ' + ', '.join(f'{v:+.5f}' for v in c3a['stage2']['quartz_k']))
print(f"           d_q {c3a['stage2']['d_quartz_um']:.2f}, d_Si {c3a['stage2']['d_Si_um']:.2f} um, loss {c3a['stage2']['loss']:.5e} "
      f"({100*(c3a['stage2']['loss']/c3a['stage1']['loss']-1):+.1f} % vs stage 1)")

[C3 stage 1 (tied) lr 0.001] Epoch=     1  loss=4.6598e-02 (k-pen 0.0e+00)  [knot 0.76 THz] quartz: n=1.98200 k=0.00710 d=1000.0um | Si: n=3.40000 k=0.00540 d=400.0um |


[C3 stage 1 (tied) lr 0.001] Epoch=  5000  loss=4.4578e-03 (k-pen 0.0e+00)  [knot 0.76 THz] quartz: n=1.96744 k=0.00552 d=992.5um | Si: n=3.42949 k=-0.00207 d=383.4um |


[C3 stage 1 (tied) lr 0.001] Epoch=  6000  loss=4.4549e-03 (k-pen 0.0e+00)  [knot 0.76 THz] quartz: n=1.96689 k=0.00547 d=992.8um | Si: n=3.43305 k=-0.00197 d=383.0um |


[C3 stage 1 (tied) lr 0.0001] Epoch=     1  loss=4.4549e-03 (k-pen 0.0e+00)  [knot 0.76 THz] quartz: n=1.96689 k=0.00547 d=992.8um | Si: n=3.43305 k=-0.00197 d=383.0um |


[C3 stage 1 (tied) lr 0.0001] Epoch=  4000  loss=4.4535e-03 (k-pen 0.0e+00)  [knot 0.76 THz] quartz: n=1.96629 k=0.00541 d=993.1um | Si: n=3.43629 k=-0.00183 d=382.7um |


[C3 stage 2 (6 knots) lr 0.001] Epoch=     1  loss=4.4535e-03 (k-pen 0.0e+00)  [knot 0.76 THz] quartz: n=1.96629 k=0.00541 d=993.1um | Si: n=3.43629 k=-0.00183 d=382.7um |


... (3 outputs left out)

[C3 stage 2 (6 knots) lr 0.0001] Epoch=  5000  loss=2.7915e-03 (k-pen 0.0e+00)  [knot 0.76 THz] quartz: n=1.96993 k=0.00405 d=991.6um | Si: n=3.41630 k=-0.00278 d=384.8um |


[C3 stage 2 (6 knots) lr 0.0001] Epoch= 10000  loss=2.7899e-03 (k-pen 0.0e+00)  [knot 0.76 THz] quartz: n=1.97025 k=0.00407 d=991.4um | Si: n=3.41495 k=-0.00285 d=384.9um |


[C3 stage 2 (6 knots) lr 0.0001] Epoch= 14000  loss=2.7893e-03 (k-pen 0.0e+00)  [knot 0.76 THz] quartz: n=1.97050 k=0.00409 d=991.3um | Si: n=3.41391 k=-0.00290 d=385.1um |



C3 (i), 169 s.  stage 1: kappa_Si -1.827e-03, kappa_q 5.410e-03, d_q 993.07, d_Si 382.68 um, loss 4.45350e-03
  stage 2: Si kappa knots -0.00276, -0.00290, -0.00222, -0.00363, -0.00262, +0.00057  (negative in band 97 %)
           Si n knots     3.4133, 3.4139, 3.4148, 3.4161, 3.4191, 3.4303
           quartz k knots +0.00004, +0.00409, +0.00672, +0.00993, +0.01186, +0.01268
           d_q 991.30, d_Si 385.06 um, loss 2.78930e-03 (-37.4 % vs stage 1)


## C3 (ii) — converged least-squares comparison: rMSE L-BFGS, tied vs six knots (λ_κ = 0)

In [3]:
A_g = torch.as_tensor(interp_basis(grid.f_np, f_knots, 'cubic'), dtype=DT)
y_t = torch.tensor(y_meas, dtype=DT)
def y_disp(th):            # th: nq(6) kq(6) ns(6) ks(6) dq ds
    nq, kq, ns, ks = (A_g @ th[6 * i: 6 * i + 6] for i in range(4))
    return simulate_dispersive(grid, x_exp, [(nq - 1j * kq, th[24]), (ns - 1j * ks, th[25])])[1][:L]
SC = torch.tensor(np.r_[np.full(6, 0.01), np.full(6, 0.001), np.full(6, 0.01), np.full(6, 0.001), 1e-6, 1e-6], dtype=DT)
def rmse_disp(z, th0):
    return torch.sqrt(torch.mean((y_disp(th0 + SC * z) - y_t) ** 2))
def tied_to_th(p):         # p = (n1, k1, d1, n2, k2, d2) -> flat knots
    return np.r_[np.full(6, p[0]), np.full(6, p[1]), np.full(6, p[3]), np.full(6, p[4]), p[2], p[5]]
t0 = time.time()
# tied model = the two-layer frequency-independent model: converged rMSE optimum from the nb3 best point
nb3j = json.load(open('../results/nb3.json'))
b3 = nb3j['best']; p_adam = np.array([b3['n1'], b3['k1'], b3['d1_um'] * 1e-6, b3['n2'], b3['k2'], b3['d2_um'] * 1e-6])
ref_t = torch.tensor(p_adam, dtype=DT)
u_t, l_tied, ev_t = lbfgs_minimize(lambda u: P.loss_torch(u * ref_t, 'rmse'), np.ones(6), max_iter=1500)
p_tied = u_t * p_adam
th_tied = torch.tensor(tied_to_th(p_tied), dtype=DT)
assert abs(torch.sqrt(torch.mean((y_disp(th_tied) - y_t) ** 2)).item() - l_tied) < 1e-9
z_d, l_disp, ev_d = lbfgs_minimize(lambda z: rmse_disp(z, th_tied), np.zeros(26), max_iter=3000)
th_d = th_tied.numpy() + SC.numpy() * z_d
gn_d, _ = grad_norm(lambda z: rmse_disp(z, th_tied), z_d)
T['C3_lbfgs_s'] = time.time() - t0
c3b = dict(tied=dict(params=dict(zip(PARAM_NAMES, p_tied)), rmse=l_tied, rmse_over_sigma=l_tied / sigma, evaluations=ev_t),
           six_knots=dict(quartz_n=th_d[0:6], quartz_k=th_d[6:12], Si_n=th_d[12:18], Si_k=th_d[18:24], d_quartz_um=th_d[24] * 1e6, d_Si_um=th_d[25] * 1e6,
                          rmse=l_disp, rmse_over_sigma=l_disp / sigma, evaluations=ev_d, grad_norm_scaled=gn_d,
                          Si_k_mean_interior_knots=float(th_d[19:23].mean()), Si_k_all_negative=bool(np.all(th_d[18:24] < 0))),
           rmse_reduction_percent=100 * (1 - l_disp / l_tied), wall_time_s=T['C3_lbfgs_s'])
print(f"C3 (ii), {T['C3_lbfgs_s']:.0f} s.  tied (frequency-independent) rMSE optimum: kappa_Si {p_tied[4]:+.4e}, n_Si {p_tied[3]:.4f}, d1 {p_tied[2]*1e6:.2f}, d2 {p_tied[5]*1e6:.2f} um, "
      f"rMSE {l_tied:.5e} ({ev_t} evaluations)")
print('   six knots: Si kappa ' + ', '.join(f'{v:+.5f}' for v in th_d[18:24]) + ';  Si n ' + ', '.join(f'{v:.4f}' for v in th_d[12:18]))
print('              quartz kappa ' + ', '.join(f'{v:+.5f}' for v in th_d[6:12]) + f";  d_q {th_d[24]*1e6:.2f}, d_Si {th_d[25]*1e6:.2f} um")
print(f"   rMSE {l_disp:.5e} ({100*(1-l_disp/l_tied):.1f} % below tied; {ev_d} evaluations, |grad| {gn_d:.1e})")
RESULTS['C3_six_knots_bilayer'] = dict(adam_protocol=c3a, converged_rmse=c3b)

C3 (ii), 44 s.  tied (frequency-independent) rMSE optimum: kappa_Si -7.2744e-03, n_Si 3.3037, d1 977.71, d2 398.14 um, rMSE 1.12398e-02 (2041 evaluations)
   six knots: Si kappa -0.00602, -0.00380, -0.00418, -0.00514, -0.00528, -0.00296;  Si n 3.3920, 3.3955, 3.3960, 3.3978, 3.3982, 3.4020
              quartz kappa -0.00060, +0.00434, +0.00812, +0.01073, +0.01290, +0.01539;  d_q 989.30, d_Si 387.26 um
   rMSE 4.84013e-03 (56.9 % below tied; 7236 evaluations, |grad| 1.0e-07)


## C4 — interface layer between quartz and Si, and a free reference delay / amplitude (frequency-independent, rMSE L-BFGS)

In [4]:
from RobustExtractor import Forward64
fwd = P.fwd
def y3(p, dg, ng):          # quartz / gap / Si, all frequency independent; gap kappa = 0
    n = torch.stack([p[0], ng, p[3]]); k = torch.stack([-p[1], torch.zeros((), dtype=DT), -p[4]]); d = torch.stack([p[2], dg, p[5]])
    return fwd(n, k, d)
def make_fun(p0, variant, dg0=None):
    p0 = torch.tensor(p0, dtype=DT)
    if variant == 'baseline':
        return lambda z: torch.sqrt(torch.mean((fwd(torch.stack([z[0], z[3]]) * p0[[0, 3]], -torch.stack([z[1], z[4]]) * p0[[1, 4]], torch.stack([z[2], z[5]]) * p0[[2, 5]]) - y_t) ** 2)), np.ones(6)
    if variant == 'tau_amp':
        def f(z):
            H = tau_amp_filter(fwd.freqs, z[6] * 10e-15, torch.exp(0.01 * z[7]))
            return torch.sqrt(torch.mean((fwd(torch.stack([z[0], z[3]]) * p0[[0, 3]], -torch.stack([z[1], z[4]]) * p0[[1, 4]], torch.stack([z[2], z[5]]) * p0[[2, 5]], H) - y_t) ** 2))
        return f, np.r_[np.ones(6), 0.0, 0.0]
    if variant == 'air_gap':
        return lambda z: torch.sqrt(torch.mean((y3(z[:6] * p0, dg0 * torch.exp(z[6]), torch.tensor(1.0, dtype=DT)) - y_t) ** 2)), np.r_[np.ones(6), 0.0]
    if variant == 'adhesive':
        return lambda z: torch.sqrt(torch.mean((y3(z[:6] * p0, dg0 * torch.exp(z[6]), 1.4 + 0.4 * torch.sigmoid(z[7])) - y_t) ** 2)), np.r_[np.ones(6), 0.0, 0.0]
def run_variant(k, variant, dg0=None):
    p0 = STARTS[k]
    f, z0 = make_fun(p0, variant, dg0)
    t0 = time.time()
    z, l, ev = lbfgs_minimize(f, z0, max_iter=1500)
    if not np.isfinite(l):
        z, l, ev = lbfgs_minimize(f, z0, max_iter=1500, lr=0.1)
    p = z[:6] * p0
    out = dict(start=k + 1, variant=variant, n1=p[0], k1=p[1], d1_um=p[2] * 1e6, n2=p[3], k2=p[4], d2_um=p[5] * 1e6, rmse=l, rmse_over_sigma=l / sigma,
               evaluations=ev, wall_s=time.time() - t0, grad_norm=grad_norm(f, z)[0], mae_logcosh=P.loss_np(p))
    if variant in ('air_gap', 'adhesive'):
        out.update(d_gap_start_um=dg0 * 1e6, d_gap_um=dg0 * np.exp(z[6]) * 1e6)
    if variant == 'adhesive':
        out['n_gap'] = 1.4 + 0.4 / (1 + np.exp(-z[7]))
    if variant == 'tau_amp':
        out.update(tau_fs=z[6] * 10.0, amplitude=float(np.exp(0.01 * z[7])))
    out['valid'] = bool(np.isfinite(l) and p[0] > 1 and p[3] > 1 and p[2] > 0 and p[5] > 0 and l < 2 * 1.124e-2 and out.get('d_gap_um', 1.0) < 1000)
    return out
t0 = time.time()
c4 = dict(baseline=[run_variant(k, 'baseline') for k in range(3)], tau_amp=[run_variant(k, 'tau_amp') for k in range(3)],
          air_gap=[run_variant(k, 'air_gap', dg) for k in range(3) for dg in (1e-6, 5e-6, 20e-6)],
          adhesive=[run_variant(k, 'adhesive', dg) for k in range(3) for dg in (1e-6, 5e-6, 20e-6)])
T['C4_s'] = time.time() - t0
l_base = min(r['rmse'] for r in c4['baseline'])
print(f"C4 ({T['C4_s']:.0f} s); baseline (two layers) best rMSE {l_base:.5e}")
print(f"{'variant':9s} {'st':>2s} {'gap0':>5s} | {'d_gap':>8s} {'n_gap':>6s} {'tau/fs':>7s} {'a':>7s} | {'n1':>7s} {'k1':>9s} {'d1':>8s} {'n2':>7s} {'k2':>10s} {'d2':>8s} | {'rMSE/sig':>8s} {'vs base':>8s} {'evals':>5s}")
for vname, rr in c4.items():
    for r in rr:
        print(f"{vname:9s} {r['start']:2d} {r.get('d_gap_start_um', float('nan')):5.0f} | {r.get('d_gap_um', float('nan')):8.3f} {r.get('n_gap', float('nan')):6.3f} "
              f"{r.get('tau_fs', float('nan')):7.1f} {r.get('amplitude', float('nan')):7.4f} | {r['n1']:7.4f} {r['k1']:9.2e} {r['d1_um']:8.2f} {r['n2']:7.4f} {r['k2']:+10.3e} {r['d2_um']:8.2f} | "
              f"{r['rmse_over_sigma']:8.3f} {100*(r['rmse']/l_base-1):+7.2f}% {r['evaluations']:5d}")
summary = {}
for vname, rr in c4.items():
    ok = [r for r in rr if r['valid']]
    bst = min(ok, key=lambda r: r['rmse'])
    summary[vname] = dict(best=bst, rmse_change_vs_baseline_percent=100 * (bst['rmse'] / l_base - 1), k2_sign_of_best=int(np.sign(bst['k2'])),
                          n_runs_with_k2_positive=int(sum(r['k2'] > 0 for r in ok)), n_valid_runs=len(ok), n_runs=len(rr),
                          n_valid_runs_at_best_within_0p1pct=int(sum(r['rmse'] <= 1.001 * bst['rmse'] for r in ok)))
RESULTS['C4_interface_layer'] = dict(loss='rMSE (least squares), L-BFGS strong Wolfe, two .step() calls of up to 1500 iterations', baseline_rmse=l_base,
                                     runs=c4, summary=summary, wall_time_s=T['C4_s'])
for vname, s_ in summary.items():
    print(f"  best {vname:9s}: kappa_Si {s_['best']['k2']:+.3e}, n_Si {s_['best']['n2']:.4f}, rMSE change {s_['rmse_change_vs_baseline_percent']:+.2f} %, "
          f"kappa_Si > 0 in {s_['n_runs_with_k2_positive']}/{s_['n_valid_runs']} valid runs ({s_['n_runs']} run; invalid = diverged to unphysical values)")

C4 (237 s); baseline (two layers) best rMSE 1.12398e-02
variant   st  gap0 |    d_gap  n_gap  tau/fs       a |      n1        k1       d1      n2         k2       d2 | rMSE/sig  vs base evals
baseline   1   nan |      nan    nan     nan     nan |  1.9968  7.78e-03   977.71  3.3037 -7.274e-03   398.14 |   38.749   +0.00%    90
baseline   2   nan |      nan    nan     nan     nan |  1.9968  7.78e-03   977.71  3.3037 -7.274e-03   398.14 |   38.749   +0.00%   196
baseline   3   nan |      nan    nan     nan     nan |  1.9968  7.78e-03   977.71  3.3037 -7.274e-03   398.14 |   38.749   +0.00%   103
tau_amp    1   nan |      nan    nan  -196.3  1.1221 |  2.0571  9.53e-03   949.00  3.5771 -1.176e-06   367.63 |   27.230  -29.73%  3750
tau_amp    2   nan |      nan    nan  -241.8  1.1422 |  2.0722  9.17e-03   942.08  3.6439 +2.716e-03   360.88 |   26.983  -30.36%   500
tau_amp    3   nan |      nan    nan  -241.8  1.1422 |  2.0722  9.17e-03   942.08  3.6439 +2.716e-03   360.88 |   26.983  -30.36

## C3 + C4 combined — six knots **and** an air gap (the Sec. 8 model) on the Sec. 5 data, rMSE L-BFGS
Warm start: the six-knot optimum of C3 (ii) with a gap of the C4 best size inserted.

In [5]:
def y_disp_gap(th):         # th: nq(6) kq(6) ns(6) ks(6) dq ds log(d_gap / 1 um)
    nq, kq, ns, ks = (A_g @ th[6 * i: 6 * i + 6] for i in range(4))
    return simulate_dispersive(grid, x_exp, [(nq - 1j * kq, th[24]), (torch.tensor(1.0 + 0j, dtype=torch.complex128), 1e-6 * torch.exp(th[26])), (ns - 1j * ks, th[25])])[1][:L]
SCg = torch.cat([SC, torch.tensor([1.0], dtype=DT)])
dg_best = summary['air_gap']['best']['d_gap_um']
results_e = []
t0 = time.time()
for dg0 in (dg_best, 1.0, 20.0):
    th0g = torch.tensor(np.r_[th_d, np.log(dg0)], dtype=DT)
    fg = lambda z: torch.sqrt(torch.mean((y_disp_gap(th0g + SCg * z) - y_t) ** 2))
    zg, lg, evg = lbfgs_minimize(fg, np.zeros(27), max_iter=2000)
    thg = th0g.numpy() + SCg.numpy() * zg
    results_e.append(dict(d_gap_start_um=dg0, d_gap_um=float(np.exp(thg[26])), quartz_n=thg[0:6], quartz_k=thg[6:12], Si_n=thg[12:18], Si_k=thg[18:24],
                          d_quartz_um=thg[24] * 1e6, d_Si_um=thg[25] * 1e6, rmse=lg, rmse_over_sigma=lg / sigma, evaluations=evg,
                          rmse_change_vs_six_knots_percent=100 * (lg / l_disp - 1), rmse_change_vs_two_layer_percent=100 * (lg / l_base - 1),
                          Si_k_all_negative=bool(np.all(thg[18:24] < 0)), Si_k_mean_interior_knots=float(thg[19:23].mean())))
    r = results_e[-1]
    print(f"six knots + gap (start {dg0:.1f} um): d_gap {r['d_gap_um']:.3f} um, d_q {r['d_quartz_um']:.2f}, d_Si {r['d_Si_um']:.2f} um; rMSE {lg:.5e} "
          f"({r['rmse_change_vs_six_knots_percent']:+.1f} % vs six knots, {r['rmse_change_vs_two_layer_percent']:+.1f} % vs two-layer flat); {evg} evaluations")
    print('    Si kappa ' + ', '.join(f'{v:+.5f}' for v in r['Si_k']) + ';  Si n ' + ', '.join(f'{v:.4f}' for v in r['Si_n']))
T['C3C4_combined_s'] = time.time() - t0
RESULTS['C3_C4_six_knots_plus_gap'] = dict(loss='rMSE, L-BFGS', runs=results_e, wall_time_s=T['C3C4_combined_s'])

six knots + gap (start 8.0 um): d_gap 6.398 um, d_q 971.79, d_Si 398.34 um; rMSE 4.64882e-03 (-4.0 % vs six knots, -58.6 % vs two-layer flat); 2433 evaluations
    Si kappa -0.01259, -0.00356, -0.00015, +0.00192, +0.00300, +0.00503;  Si n 3.3220, 3.3216, 3.3194, 3.3177, 3.3158, 3.3162


six knots + gap (start 1.0 um): d_gap 6.393 um, d_q 971.81, d_Si 398.34 um; rMSE 4.64882e-03 (-4.0 % vs six knots, -58.6 % vs two-layer flat); 5002 evaluations
    Si kappa -0.01262, -0.00356, -0.00016, +0.00189, +0.00286, +0.00448;  Si n 3.3221, 3.3216, 3.3195, 3.3178, 3.3159, 3.3165


six knots + gap (start 20.0 um): d_gap 6.388 um, d_q 971.83, d_Si 398.31 um; rMSE 4.64882e-03 (-4.0 % vs six knots, -58.6 % vs two-layer flat); 3399 evaluations
    Si kappa -0.01260, -0.00356, -0.00016, +0.00189, +0.00293, +0.00472;  Si n 3.3222, 3.3218, 3.3196, 3.3180, 3.3161, 3.3167


# Loop-2 check C16 — optical-thickness products of every Sec. 5 model variant
n_q d_q, n_Si d_Si, d_q + d_Si and Σ(n−1)d.  For the six-knot variants n is taken at 1 THz from the cubic spline
(phase index) and the group index n_g = n + f dn/df at 1 THz is also given.  Two-layer Adam / softplus from nb3.json,
L-BFGS on Eq. (4) from nb4.json; all other variants from this notebook.

In [6]:
from scipy.interpolate import CubicSpline
nb4j = json.load(open('../results/nb4.json'))
def prod_flat(n1, d1_um, n2, d2_um, **extra):
    return dict(nq_dq_um=n1 * d1_um, nSi_dSi_um=n2 * d2_um, dq_plus_dSi_um=d1_um + d2_um, sum_n_minus_1_d_um=(n1 - 1) * d1_um + (n2 - 1) * d2_um,
                n_q=n1, d_q_um=d1_um, n_Si=n2, d_Si_um=d2_um, **extra)
def prod_knots(nq_k, d1_um, ns_k, d2_um, f=1.0, **extra):
    out = {}
    for tag, nk in (('q', nq_k), ('Si', ns_k)):
        cs = CubicSpline(f_knots, np.asarray(nk, float), bc_type='natural')
        out[tag] = (float(cs(f)), float(cs(f) + f * cs(f, 1)))
    (nq, ngq), (ns, ngs) = out['q'], out['Si']
    return dict(nq_dq_um=nq * d1_um, nSi_dSi_um=ns * d2_um, dq_plus_dSi_um=d1_um + d2_um, sum_n_minus_1_d_um=(nq - 1) * d1_um + (ns - 1) * d2_um,
                group_nq_dq_um=ngq * d1_um, group_nSi_dSi_um=ngs * d2_um, group_sum_n_minus_1_d_um=(ngq - 1) * d1_um + (ngs - 1) * d2_um,
                n_q_1THz=nq, ng_q_1THz=ngq, n_Si_1THz=ns, ng_Si_1THz=ngs, d_q_um=d1_um, d_Si_um=d2_um, index_at='1 THz', **extra)
c16 = {}
for k, r in enumerate(nb3j['fits']):
    c16[f'two_layer_adam_start{k+1}'] = prod_flat(r['n1'], r['d1_um'], r['n2'], r['d2_um'], loss='Eq. (4)')
cs_ = nb3j['constrained_softplus']; c16['two_layer_softplus_start1'] = prod_flat(cs_['n1'], cs_['d1_um'], cs_['n2'], cs_['d2_um'], loss='Eq. (4)')
for k, r in enumerate(nb4j['lbfgs']):
    f_ = r['final']; c16[f'two_layer_lbfgs_eq4_start{k+1}'] = prod_flat(f_['n1'], f_['d1_um'], f_['n2'], f_['d2_um'], loss='Eq. (4)')
for vname in ('baseline', 'air_gap', 'adhesive', 'tau_amp'):
    bst = summary[vname]['best']
    extra = dict(loss='rMSE')
    if 'd_gap_um' in bst: extra.update(d_gap_um=bst['d_gap_um'], total_thickness_um=bst['d1_um'] + bst['d_gap_um'] + bst['d2_um'])
    if 'n_gap' in bst: extra.update(n_gap=bst['n_gap'])
    if 'tau_fs' in bst: extra.update(tau_fs=bst['tau_fs'], amplitude=bst['amplitude'])
    c16[{'baseline': 'two_layer_rmse', 'air_gap': 'gap_rmse', 'adhesive': 'adhesive_rmse', 'tau_amp': 'two_layer_tau_amp_rmse'}[vname]] = prod_flat(bst['n1'], bst['d1_um'], bst['n2'], bst['d2_um'], **extra)
s2 = c3a['stage2']
c16['six_knot_adam_eq4'] = prod_knots(s2['quartz_n'], s2['d_quartz_um'], s2['Si_n'], s2['d_Si_um'], loss='Eq. (4), lambda 0')
c16['six_knot_rmse'] = prod_knots(th_d[0:6], th_d[24] * 1e6, th_d[12:18], th_d[25] * 1e6, loss='rMSE')
e0 = min(results_e, key=lambda r: r['rmse'])
c16['six_knot_plus_gap_rmse'] = prod_knots(e0['quartz_n'], e0['d_quartz_um'], e0['Si_n'], e0['d_Si_um'], loss='rMSE', d_gap_um=e0['d_gap_um'])
print(f"{'variant':30s} {'n_q d_q':>9s} {'n_Si d_Si':>9s} {'d_q+d_Si':>9s} {'S(n-1)d':>9s} | group n_q d_q, n_Si d_Si (1 THz)")
for k_, r in c16.items():
    print(f"{k_:30s} {r['nq_dq_um']:9.1f} {r['nSi_dSi_um']:9.1f} {r['dq_plus_dSi_um']:9.1f} {r['sum_n_minus_1_d_um']:9.1f}"
          + (f" | {r['group_nq_dq_um']:9.1f} {r['group_nSi_dSi_um']:9.1f}" if 'group_nq_dq_um' in r else ''))
RESULTS['loop2_checks'] = dict(date='2026-11-24', source='reviews/loop2_domain.md',
                               C16_optical_thickness_products=dict(units='um', note='two-layer Adam/softplus from nb3.json, L-BFGS Eq. (4) from nb4.json; '
                                                                   'gap/adhesive/tau-free = best valid C4 runs; six-knot values: phase index at 1 THz, group index n + f dn/df at 1 THz',
                                                                   variants=c16))

variant                          n_q d_q n_Si d_Si  d_q+d_Si   S(n-1)d | group n_q d_q, n_Si d_Si (1 THz)
two_layer_adam_start1             1952.6    1315.0    1375.7    1892.0
two_layer_adam_start2             1952.7    1315.0    1375.7    1892.0
two_layer_adam_start3             1952.6    1315.0    1375.7    1891.9
two_layer_softplus_start1         1952.8    1314.8    1375.7    1891.9
two_layer_lbfgs_eq4_start1        1952.8    1314.6    1375.7    1891.8
two_layer_lbfgs_eq4_start2        1952.9    1314.6    1375.7    1891.8
two_layer_lbfgs_eq4_start3        1952.8    1314.6    1375.7    1891.8
two_layer_rmse                    1952.3    1315.3    1375.8    1891.8
gap_rmse                          1931.4    1324.8    1367.7    1888.4
adhesive_rmse                     1929.1    1322.9    1366.0    1886.0
two_layer_tau_amp_rmse            1952.2    1315.0    1303.0    1964.3
six_knot_adam_eq4                 1954.3    1314.7    1376.4    1892.6 |    1955.7    1315.3
six_knot_rmse       

In [7]:
RESULTS['timings_s'] = T
json.dump(ps.to_jsonable(RESULTS), open('../results/nb6.json', 'w'), indent=1)
print('wrote ../results/nb6.json')

wrote ../results/nb6.json
